# Network perimeter analysis

Compute 1-link and 2-link network containment perimeters from USAMMv3 dairy cattle movement networks.

Three 2-link methods:
- **Unconditional**: fixed 1-link source set used in every network
- **Approach A (chain-aware)**: only hop forward from intermediaries active in that network; destination must appear in ≥threshold of all 1000 networks
- **Approach B (per-edge conditional)**: for each (intermediary, destination) pair, check if the link appears in ≥threshold of networks *where that intermediary is active*

In [1]:
import pandas as pd
import numpy as np
import yaml
from pathlib import Path
from collections import Counter, defaultdict

PROJ = Path('../..').resolve()
CONFIG = PROJ / 'analysis' / 'simulations' / 'config.yaml'
NET_DIR = PROJ / 'analysis' / 'H5N1-Simulations-and-Analysis' / 'data' / 'raw' / 'USAMMv3_cattle_networks' / 'dairy'
FIG_DIR = PROJ / 'paper' / 'final' / 'figures'

with open(CONFIG) as f:
    cfg = yaml.safe_load(f)

panhandle = set(cfg['county_sets']['panhandle'])
scenario = pd.read_csv(PROJ / 'paper' / 'figures' / 'all_county_origins' / 'scenario_summary.csv')
dairy_fips = set(scenario['seed_county'].unique())
fips_lookup = pd.read_csv(PROJ / 'analysis' / 'H5N1-Simulations-and-Analysis' / 'data' / 'raw' / 'US_FIPS_Codes_formatted.csv')
fips_name = dict(zip(fips_lookup['fips'], fips_lookup['county'] + ', ' + fips_lookup['state']))

network_files = sorted(NET_DIR.glob('dairy_network_*.network'))
n = len(network_files)
print(f'Networks: {n}')
print(f'Panhandle: {len(panhandle)} counties')
print(f'Dairy counties: {len(dairy_fips)}')

Networks: 1000
Panhandle: 14 counties
Dairy counties: 2504


In [2]:
# 1-hop counts: for each dairy county, how many networks have a panhandle -> county edge?
hop1_counts = Counter()
for i, nf in enumerate(network_files):
    df = pd.read_csv(nf, sep='\t', usecols=['oCountyId', 'dCountyId'])
    mask = df['oCountyId'].isin(panhandle) & df['dCountyId'].isin(dairy_fips) & ~df['dCountyId'].isin(panhandle)
    for d in df.loc[mask, 'dCountyId'].unique():
        hop1_counts[d] += 1
    if (i + 1) % 200 == 0:
        print(f'  {i+1}/{n}', flush=True)

print(f'\nTotal unique 1-hop destinations: {len(hop1_counts)}')

  200/1000
  400/1000
  600/1000
  800/1000
  1000/1000

Total unique 1-hop destinations: 2277


In [3]:
# 1-hop sets at each threshold
thresholds = [0.90, 0.95, 0.99, 1.00]

h1_sets = {}
net_1hops = {}
for t in thresholds:
    cutoff = n * t
    h1 = {f for f, c in hop1_counts.items() if c >= cutoff}
    h1_sets[t] = h1
    net_1hops[t] = panhandle | h1

print(f'{"Threshold":<12} {"Destinations":<15} {"Total (+ panhandle)"}')
print('-' * 45)
for t in thresholds:
    print(f'{t:.0%}{"":<9} {len(h1_sets[t]):<15} {len(net_1hops[t])}')

Threshold    Destinations    Total (+ panhandle)
---------------------------------------------
90%          52              66
95%          38              52
99%          27              41
100%          15              29


In [4]:
# 2-hop computation: unconditional, chain-aware (A), and per-edge conditional (B)
# All thresholds computed in a single pass through the networks

# Unconditional: use fixed 1-hop set as source in every network
uncond_counts = {t: Counter() for t in thresholds}

# Approach A: chain-aware (only active intermediaries)
a_counts = {t: Counter() for t in thresholds}

# Approach B: per-edge conditional
b_pairs = {t: defaultdict(int) for t in thresholds}

for i, nf in enumerate(network_files):
    df = pd.read_csv(nf, sep='\t', usecols=['oCountyId', 'dCountyId'])

    for t in thresholds:
        # Unconditional: fixed source set
        mask_unc = df['oCountyId'].isin(net_1hops[t]) & df['dCountyId'].isin(dairy_fips) & ~df['dCountyId'].isin(net_1hops[t])
        for d in df.loc[mask_unc, 'dCountyId'].unique():
            uncond_counts[t][d] += 1

        # Chain-aware: only active intermediaries
        mask_h1 = df['oCountyId'].isin(panhandle) & df['dCountyId'].isin(h1_sets[t])
        active_h1 = set(df.loc[mask_h1, 'dCountyId'].unique())
        if not active_h1:
            continue

        mask_h2 = df['oCountyId'].isin(active_h1) & df['dCountyId'].isin(dairy_fips) & ~df['dCountyId'].isin(net_1hops[t])
        h2_edges = df.loc[mask_h2, ['oCountyId', 'dCountyId']].drop_duplicates()

        for d in h2_edges['dCountyId'].unique():
            a_counts[t][d] += 1

        for _, row in h2_edges.iterrows():
            b_pairs[t][(int(row['oCountyId']), int(row['dCountyId']))] += 1

    if (i + 1) % 200 == 0:
        print(f'  {i+1}/{n}', flush=True)

print('Done.')

  200/1000
  400/1000
  600/1000
  800/1000
  1000/1000
Done.


In [5]:
# Compute final sets and display results
results = []

for t in thresholds:
    cutoff = n * t

    # Unconditional
    u_2hop = {d for d, c in uncond_counts[t].items() if c >= cutoff}
    u_total = net_1hops[t] | u_2hop

    # Approach A
    a_2hop = {d for d, c in a_counts[t].items() if c >= cutoff}
    a_total = net_1hops[t] | a_2hop

    # Approach B
    b_dest = set()
    for (i_county, d), pair_count in b_pairs[t].items():
        n_i = hop1_counts[i_county]
        if pair_count / n_i >= t:
            b_dest.add(d)
    b_total = net_1hops[t] | b_dest

    results.append({
        'threshold': t,
        'hop1': len(net_1hops[t]),
        'uncond': len(u_total),
        'chain_aware': len(a_total),
        'conditional': len(b_total),
        'uncond_set': u_total,
        'chain_set': a_total,
        'cond_set': b_total,
    })

print(f'{"Threshold":<12} {"1-hop":<8} {"Unconditional":<15} {"A: Chain-aware":<17} {"B: Conditional"}')
print('=' * 65)
for r in results:
    print(f'{r["threshold"]:.0%}{"":<9} {r["hop1"]:<8} {r["uncond"]:<15} {r["chain_aware"]:<17} {r["conditional"]}')

Threshold    1-hop    Unconditional   A: Chain-aware    B: Conditional
90%          66       442             381               195
95%          52       314             260               132
99%          41       189             154               102
100%          29       73              55                47


In [6]:
# Subset relationship checks
print('Subset checks (B ⊂ A ⊂ Unconditional?):\n')
for r in results:
    t = r['threshold']
    b_sub_a = r['cond_set'].issubset(r['chain_set'])
    a_sub_u = r['chain_set'].issubset(r['uncond_set'])
    print(f'{t:.0%}: B ⊂ A? {b_sub_a}  |  A ⊂ Unconditional? {a_sub_u}')

# Cross-threshold subset checks for approach B
print('\nApproach B cross-threshold subsets (stricter ⊂ looser?):')
for i in range(len(results) - 1, 0, -1):
    strict = results[i]
    loose = results[i-1]
    is_sub = strict['cond_set'].issubset(loose['cond_set'])
    print(f'  B@{strict["threshold"]:.0%} ({strict["conditional"]}) ⊂ B@{loose["threshold"]:.0%} ({loose["conditional"]})? {is_sub}')

Subset checks (B ⊂ A ⊂ Unconditional?):

90%: B ⊂ A? True  |  A ⊂ Unconditional? True
95%: B ⊂ A? True  |  A ⊂ Unconditional? True
99%: B ⊂ A? True  |  A ⊂ Unconditional? True
100%: B ⊂ A? True  |  A ⊂ Unconditional? True

Approach B cross-threshold subsets (stricter ⊂ looser?):
  B@100% (47) ⊂ B@99% (102)? True
  B@99% (102) ⊂ B@95% (132)? True
  B@95% (132) ⊂ B@90% (195)? True


In [7]:
# Compare with geographic perimeters for context
geo_ring2 = len(cfg['county_sets']['geo_ring2'])
geo_ring4 = len(cfg['county_sets']['geo_ring4'])

print('All perimeters by county count:\n')
perimeters = [
    ('Panhandle', len(panhandle), 'Origin'),
]
for r in sorted(results, key=lambda x: -x['threshold']):
    t = r['threshold']
    perimeters.append((f'1-link {int(t*100)}%', r['hop1'], 'Network'))

perimeters.append(('Geo ring 2', geo_ring2, 'Geographic'))

for r in sorted(results, key=lambda x: -x['threshold']):
    t = r['threshold']
    perimeters.append((f'2-link {int(t*100)}% (B)', r['conditional'], f'Network (conditional)'))

perimeters.append(('Geo ring 4', geo_ring4, 'Geographic'))

for r in sorted(results, key=lambda x: -x['threshold']):
    t = r['threshold']
    perimeters.append((f'2-link {int(t*100)}% (uncond)', r['uncond'], f'Network (unconditional)'))

perimeters.sort(key=lambda x: x[1])

print(f'{"Perimeter":<30} {"Counties":<10} {"Strategy"}')
print('-' * 65)
for name, count, strategy in perimeters:
    print(f'{name:<30} {count:<10} {strategy}')

All perimeters by county count:

Perimeter                      Counties   Strategy
-----------------------------------------------------------------
Panhandle                      14         Origin
1-link 100%                    29         Network
1-link 99%                     41         Network
Geo ring 2                     47         Geographic
2-link 100% (B)                47         Network (conditional)
1-link 95%                     52         Network
1-link 90%                     66         Network
2-link 100% (uncond)           73         Network (unconditional)
2-link 99% (B)                 102        Network (conditional)
Geo ring 4                     111        Geographic
2-link 95% (B)                 132        Network (conditional)
2-link 99% (uncond)            189        Network (unconditional)
2-link 90% (B)                 195        Network (conditional)
2-link 95% (uncond)            314        Network (unconditional)
2-link 90% (uncond)            442       

In [8]:
# Approach B: near-threshold destinations at each threshold
for t in thresholds:
    print(f'\n=== Approach B near-threshold at {t:.0%} ===')
    dest_best = {}
    for (i_county, d), pair_count in b_pairs[t].items():
        n_i = hop1_counts[i_county]
        cond = pair_count / n_i
        if d not in dest_best or cond > dest_best[d][0]:
            dest_best[d] = (cond, i_county, pair_count, n_i)

    near = [(d, v) for d, v in dest_best.items() if (t - 0.05) <= v[0] < (t + 0.02)]
    near.sort(key=lambda x: -x[1][0])
    for d, (cond, via, pc, ni) in near[:8]:
        status = 'IN' if cond >= t else 'OUT'
        print(f'  [{status}] {fips_name.get(d, d)}: {pc}/{ni} = {cond:.3f} via {fips_name.get(via, via)}')


=== Approach B near-threshold at 90% ===
  [IN] Mille Lacs, Minnesota: 844/918 = 0.919 via Stearns, Minnesota
  [IN] Cheyenne, Kansas: 918/1000 = 0.918 via Weld, Colorado
  [IN] Faribault, Minnesota: 916/999 = 0.917 via Sioux, Iowa
  [IN] Mitchell, Iowa: 913/999 = 0.914 via Sioux, Iowa
  [IN] Lake, South Dakota: 912/999 = 0.913 via Sioux, Iowa
  [IN] Le Sueur, Minnesota: 838/918 = 0.913 via Stearns, Minnesota
  [IN] Douglas, South Dakota: 904/999 = 0.905 via Sioux, Iowa
  [IN] Pine, Minnesota: 830/918 = 0.904 via Stearns, Minnesota

=== Approach B near-threshold at 95% ===
  [IN] Kossuth, Iowa: 968/999 = 0.969 via Sioux, Iowa
  [IN] Los Angeles, California: 967/1000 = 0.967 via Tulare, California
  [IN] Millard, Utah: 965/1000 = 0.965 via Tulare, California
  [IN] Renville, Minnesota: 962/999 = 0.963 via Sioux, Iowa
  [IN] Hamilton, Texas: 962/1000 = 0.962 via Erath, Texas
  [IN] Yakima, Washington: 962/1000 = 0.962 via Tulare, California
  [IN] Arapahoe, Colorado: 958/1000 = 0.958 vi

In [9]:
import json

# Save all computed sets for downstream use
output = {'panhandle': sorted([int(x) for x in panhandle])}

for r in results:
    t = int(r['threshold'] * 100)
    output[f'net{t}_1hop'] = sorted([int(x) for x in net_1hops[r['threshold']]])
    output[f'net{t}_2hop_unconditional'] = sorted([int(x) for x in r['uncond_set']])
    output[f'net{t}_2hop_chain_aware'] = sorted([int(x) for x in r['chain_set']])
    output[f'net{t}_2hop_conditional'] = sorted([int(x) for x in r['cond_set']])

out_path = PROJ / 'analysis' / 'simulations' / 'computed_network_perimeters.json'
# with open(out_path, 'w') as f:
#     json.dump(output, f, indent=2)

# print(f'Saved to {out_path}')
print(f'Keys: {list(output.keys())}')

Keys: ['panhandle', 'net90_1hop', 'net90_2hop_unconditional', 'net90_2hop_chain_aware', 'net90_2hop_conditional', 'net95_1hop', 'net95_2hop_unconditional', 'net95_2hop_chain_aware', 'net95_2hop_conditional', 'net99_1hop', 'net99_2hop_unconditional', 'net99_2hop_chain_aware', 'net99_2hop_conditional', 'net100_1hop', 'net100_2hop_unconditional', 'net100_2hop_chain_aware', 'net100_2hop_conditional']
